# **RICELEAF DISEASE PREDICTION**

The dataset contains 120 images of rice leaves categorized into three disease classes:

*   Leaf smut
*   Brown spot
*   Bacterial leaf blight

Each class contains 40 images, making the dataset balanced.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("CNN.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

img = image.load_img("cnn/Leaf smut/DSC_0308.JPG")
plt.imshow(img)
plt.axis('off')

In [ ]:
import os

os.listdir()

In [ ]:
os.listdir('cnn')

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (150, 150)
batch_size = 10

#Our data is too small for training the model, so using data augmentation to fake more datas
train_datagen = ImageDataGenerator(
    rescale=1./255,          # normalize pixel values
    validation_split=0.2,    # validation data (used during training)

    rotation_range=30,       # rotate image
    zoom_range=0.3,          # zoom in/out
    horizontal_flip=True    # flip image
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "cnn",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    "cnn",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# **DATA AUGMENTATION**

Data augmentation is used to artificially increase the size and diversity of the dataset by applying transformations to existing images.

The following augmentation techniques were applied using ImageDataGenerator:


*   Rescaling (1./255): Normalizes pixel values from 0–255 to 0–1, helping the model learn efficiently

*   Validation Split (0.2): Reserves 20% of data for validation during training.

*   Rotation (10 degrees): Slightly rotates images to simulate different orientations.
*   Zoom (0.1): Zooms images in and out to create variation.


*   Horizontal Flip: Flips images to increase diversity

Importance:

*   The dataset is very small (only 120 images)

*   It helps reduce overfitting
*   It improves model generalization


*   It simulates real-world variations in leaf images

Here, in this project, Data augmentation improves model performance by exposing the model to multiple variations of the same image, thereby helping it learn more robust and generalized features.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping  #EarlyStopping ensures that the model retains the weights from the epoch with the lowest validation loss.

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model1 = Sequential([
    Input(shape=(150,150,3)),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(),

    Flatten(),

    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
# Model training
from tensorflow.keras.optimizers import Adam

model1.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model1.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop]
)

In [ ]:
# Model structure check

model1.summary()

In [ ]:
# Accuracy
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.legend()
plt.title("Accuracy")
plt.show()

# Loss
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title("Loss")
plt.show()

In [ ]:
# Both training and validation accuracy improved while loss decreased, indicating effective learning and better model generalization

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

y_pred = model1.predict(val_data)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = val_data.classes

print(classification_report(y_true, y_pred_classes))

In [ ]:
# Test with single image

In [ ]:
os.listdir("cnn")

In [ ]:
os.listdir("cnn/Bacterial leaf blight")

In [ ]:
from tensorflow.keras.preprocessing import image

img_path = "cnn/Bacterial leaf blight/DSC_0377.JPG"

img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model1.predict(img_array)

classes = list(train_data.class_indices.keys())
print("Prediction:", classes[np.argmax(prediction)])

Challenges:



*   Small dataset size (120 images)
*   High similarity between disease patterns

*   Risk of overfitting

In [ ]:
# MobileNet for better solution

In [ ]:
# Create Train & Validation Data

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Load pre-trained MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze MobileNetV2 layers
base_model.trainable = False

# Add custom classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(3, activation='softmax')(x)

model2 = Model(
    inputs=base_model.input,
    outputs=outputs
)

In [ ]:
model2.summary()

In [ ]:
# Check Class Distribution & Create Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

In [ ]:
# Early Stopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
# Compile Model

In [ ]:
from tensorflow.keras.optimizers import Adam

model2.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

img_size = (224, 224)
batch_size = 10

train_datagen_mobilenet = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,

    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True
)

val_datagen_mobilenet = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_data_mobilenet = train_datagen_mobilenet.flow_from_directory(
    "cnn",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data_mobilenet = val_datagen_mobilenet.flow_from_directory(
    "cnn",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

In [ ]:
print(train_data_mobilenet.image_shape)

In [ ]:
# Train Model

In [ ]:
history = model2.fit(
    train_data_mobilenet,
    validation_data=val_data_mobilenet,
    epochs=20,
    class_weight=class_weights,
    callbacks=[early_stop]
)

In [ ]:
# Evaluate Model

In [ ]:
loss, accuracy = model2.evaluate(val_data)

print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

In [ ]:
# Classification Report

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predictions
y_pred = model2.predict(val_data_mobilenet)
y_pred_classes = np.argmax(y_pred, axis=1)

# Actual labels
y_true = val_data_mobilenet.classes

# Classification Report
print(classification_report(
    y_true,
    y_pred_classes,
    target_names=list(val_data_mobilenet.class_indices.keys())
))

*   Bacterial Leaf Blight achieved perfect performance with a precision, recall, and F1-score of 1.00, indicating all samples were correctly classified.
*   Brown Spot achieved an F1-score of 0.80, with a small number of samples misclassified.
*   Leaf Smut achieved a recall of 0.86 and an F1-score of 0.80, demonstrating strong detection capability.

The model achieved an overall accuracy of 87%, with balanced performance across all three disease classes as reflected by the macro-average F1-score of 0.87.

Overall, the classification report indicates that the model can effectively distinguish between different rice leaf diseases with high reliability.

In [ ]:
# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=val_data.class_indices.keys(),
    yticklabels=val_data.class_indices.keys()
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

The MobileNetV2 transfer learning approach significantly improved classification performance compared to the custom CNN model.

The trained **MobileNetV2 model achieved 86.96% validation accuracy** and demonstrated strong classification performance across all three rice leaf disease categories. The model perfectly identified Bacterial Leaf Blight and achieved reliable detection of Brown Spot and Leaf Smut, highlighting the effectiveness of transfer learning for automated rice disease classification.

In [ ]:
import joblib

# Save trained MobileNetV2 model
model2.save("rice_disease_model.keras")

# Save class names
class_names = list(train_data_mobilenet.class_indices.keys())
joblib.dump(class_names, "class_names.pkl")

print("Files saved successfully!")